In [1]:
import os
import gc
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from docx import Document
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score
import traceback
import warnings
warnings.filterwarnings('ignore')

# GPU加速库
import torch
import cupy as cp
from cuml.ensemble import RandomForestClassifier as cuRandomForestClassifier
from cuml.linear_model import LogisticRegression as cuLogisticRegression
from cuml.svm import SVC as cuSVC
from cuml.neighbors import KNeighborsClassifier as cuKNeighborsClassifier
from cuml.naive_bayes import GaussianNB as cuGaussianNB  # cuML有GaussianNB
from cuml import train_test_split as cu_train_test_split
import xgboost as xgb
from lightgbm import LGBMClassifier
import tensorflow as tf
from scikeras.wrappers import KerasClassifier
from sklearn.ensemble import VotingClassifier  # Ensemble用CPU版本，但内部模型用GPU

# 新增：PyTorch MLP模型（GPU版本）
import torch.nn as nn
import torch.optim as optim

class PyTorchMLP(nn.Module):
    """PyTorch多层感知机模型（GPU加速版），用于替代cuML的MLP"""

    def __init__(self, input_size, hidden_sizes=[64, 32], output_size=1, 
                 task_type='classification', dropout_rate=0.1):
        super(PyTorchMLP, self).__init__()
        self.task_type = task_type
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # 构建网络层
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        
        # 根据任务类型添加输出层激活函数
        if task_type == 'classification':
            layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers).to(self.device)
        
    def forward(self, x):
        return self.network(x)
    
    def fit(self, X, y, epochs=10, batch_size=32, lr=0.001):
        """训练模型"""
        self.train()
        
        # 转换数据为PyTorch张量
        if not isinstance(X, torch.Tensor):
            X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        else:
            X_tensor = X.to(self.device)
            
        if not isinstance(y, torch.Tensor):
            y_tensor = torch.tensor(y, dtype=torch.float32, device=self.device)
        else:
            y_tensor = y.to(self.device)
        
        # 确保输出维度匹配
        if self.task_type == 'classification':
            y_tensor = y_tensor.view(-1, 1)
        
        # 定义损失函数和优化器
        if self.task_type == 'classification':
            criterion = nn.BCELoss()
        else:
            criterion = nn.MSELoss()
        
        optimizer = optim.Adam(self.parameters(), lr=lr)
        
        # 训练循环
        dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        for epoch in range(epochs):
            epoch_loss = 0.0
            for batch_X, batch_y in dataloader:
                optimizer.zero_grad()
                outputs = self(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            if (epoch + 1) % 20 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(dataloader):.4f}")
    
    def predict_proba(self, X):
        """预测概率（仅分类任务）"""
        if self.task_type != 'classification':
            raise ValueError("predict_proba仅适用于分类任务")
        
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
                
            return self(X_tensor).cpu().numpy()

# 设置显示选项
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
np.set_printoptions(threshold=np.inf)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("脚本开始执行...")

# 数据路径
data_path = "l3dml_data_digital.dta"

# 定义控制变量组
control_groups = {
    1: [
        'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
        'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
        'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
        'intelligence', 'person_status', 'party', 'workasso',
        'fid','pid','year'
    ],
    2: [
        'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
        'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
        'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
        'intelligence', 'person_status', 'party', 'workasso',
        'fid','pid','year',
        'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
        'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq',  'familysize_sq', 
        'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq', 
        'intelligence_sq', 'person_status_sq'
    ],
    3: [
        'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
        'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
        'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
        'intelligence', 'person_status', 'party', 'workasso',
        'fid','pid','year',
        'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
        'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq',  'familysize_sq', 
        'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq', 
        'intelligence_sq', 'person_status_sq', 
        'ln_total_asset_cub', 'impro_sasti_cub', 'Child_cub',
        'lifefire_cub', 'age_cub', 'ln_renqing_cub', 'cfpsedu_cub',  'familysize_cub', 
        'Oldage_cub', 'health_cub', 'marriage_cub', 'ln_finc_cub', 'exercise_cub', 
        'intelligence_cub', 'person_status_cub'
    ]
}

# 定义折数（尽管IPW不直接用折，但为了与基准一致，模拟分组）
folds = [2, 5]  # 宏观区分，但IPW不需交叉验证残差，这里仅用于分组

# 自变量和因变量
treatment = 'mobile_use'
outcome = 'entrepreneurship'

# 输出目录
output_dir = "D:\\my-rapids-project\\ipw_results"
os.makedirs(output_dir, exist_ok=True)

# GPU清理函数
def clear_gpu_resources():
    gc.collect()
    time.sleep(0.5)
    cp._default_memory_pool.free_all_blocks()
    cp.get_default_memory_pool().free_all_blocks()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# 倾向评分估计函数（GPU版本）
def get_pscore_rf(X, t):
    print("正在训练 cuML Random Forest...")
    X_cu = cp.array(X)
    t_cu = cp.array(t)
    param_dist = {'n_estimators': [100, 200, 300, 400], 'max_depth': [10, 20, 30, None],
                  'min_samples_split': [2, 5, 10, 20], 'min_samples_leaf': [1, 2, 4]}
    rf = cuRandomForestClassifier(random_state=42)
    # cuML不支持RandomizedSearchCV，用简单fit
    rf.fit(X_cu, t_cu)
    return cp.asnumpy(rf.predict_proba(X_cu)[:, 1])

def get_pscore_xgb(X, t):
    print("正在训练 XGBoost (GPU)...")
    params = {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1, 'tree_method': 'gpu_hist', 'random_state': 42}
    xgb_clf = xgb.XGBClassifier(**params)
    xgb_clf.fit(X, t)
    return xgb_clf.predict_proba(X)[:, 1]

def get_pscore_lgbm(X, t):
    print("正在训练 LightGBM (CPU)...")
    params = {
        'n_estimators': 1000, 'max_depth': -1, 'learning_rate': 0.05, 'num_leaves': 31,
        'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        'reg_alpha': 0.1, 'reg_lambda': 0.1, 'random_state': 42, 'device': 'cpu', 'verbose': -1
    }
    lgbm = LGBMClassifier(**params)
    lgbm.fit(X, t)
    return lgbm.predict_proba(X)[:, 1]

def get_pscore_logistic(X, t):
    print("正在训练 cuML Logistic Regression...")
    X_cu = cp.array(X)
    t_cu = cp.array(t)
    logistic = cuLogisticRegression()
    logistic.fit(X_cu, t_cu)
    return cp.asnumpy(logistic.predict_proba(X_cu)[:, 1])

def get_pscore_svm(X, t):
    print("正在训练 cuML SVM...")
    X_cu = cp.array(X)
    t_cu = cp.array(t)
    svm = cuSVC(probability=True, random_state=42)
    svm.fit(X_cu, t_cu)
    return cp.asnumpy(svm.predict_proba(X_cu)[:, 1])

def get_pscore_knn(X, t):
    print("正在训练 cuML KNN...")
    X_cu = cp.array(X)
    t_cu = cp.array(t)
    knn = cuKNeighborsClassifier()
    knn.fit(X_cu, t_cu)
    return cp.asnumpy(knn.predict_proba(X_cu)[:, 1])

def get_pscore_dt(X, t):
    print("正在训练 Decision Tree (使用RF单树模拟)...")
    # cuML无单DT，用RF n=1模拟
    rf = cuRandomForestClassifier(n_estimators=1, random_state=42)
    X_cu = cp.array(X)
    t_cu = cp.array(t)
    rf.fit(X_cu, t_cu)
    return cp.asnumpy(rf.predict_proba(X_cu)[:, 1])

def get_pscore_nb(X, t):
    print("正在训练 cuML Naive Bayes...")
    X_cu = cp.array(X)
    t_cu = cp.array(t)
    nb = cuGaussianNB()
    nb.fit(X_cu, t_cu)
    return cp.asnumpy(nb.predict_proba(X_cu)[:, 1])

def get_pscore_mlp(X, t):
    print("正在训练 Neural Network (PyTorch GPU)...")
    input_size = X.shape[1]
    mlp = PyTorchMLP(input_size=input_size, task_type='classification')
    mlp.fit(X, t)
    return mlp.predict_proba(X)[:, 1]  # 正类概率

# ATT和ATE计算（相同）
def calculate_att_ate(data, treatment_col, outcome_col, pscore_col):
    data['weight_att'] = np.where(data[treatment_col] == 1, 1, data[pscore_col] / (1 - data[pscore_col]))
    data['weight_ate'] = np.where(data[treatment_col] == 1, 1 / data[pscore_col], 1 / (1 - data[pscore_col]))
    
    treated_outcome = data[data[treatment_col] == 1][outcome_col]
    control_data = data[data[treatment_col] == 0]
    control_weighted_outcome_att = np.average(control_data[outcome_col], weights=control_data['weight_att'])
    att = treated_outcome.mean() - control_weighted_outcome_att
    var_treated_att = treated_outcome.var()
    n_treated = len(treated_outcome)
    weighted_var_control_att = np.average((control_data[outcome_col] - control_weighted_outcome_att)**2, 
                                         weights=control_data['weight_att'])
    se_att = np.sqrt(var_treated_att / n_treated + weighted_var_control_att / len(control_data))
    ci_lower_att = att - 1.96 * se_att
    ci_upper_att = att + 1.96 * se_att
    
    treated_weighted_outcome_ate = np.average(data[data[treatment_col] == 1][outcome_col], 
                                             weights=data[data[treatment_col] == 1]['weight_ate'])
    control_weighted_outcome_ate = np.average(data[data[treatment_col] == 0][outcome_col], 
                                             weights=data[data[treatment_col] == 0]['weight_ate'])
    ate = treated_weighted_outcome_ate - control_weighted_outcome_ate
    var_treated_ate = np.average((data[data[treatment_col] == 1][outcome_col] - treated_weighted_outcome_ate)**2, 
                                 weights=data[data[treatment_col] == 1]['weight_ate'])
    var_control_ate = np.average((data[data[treatment_col] == 0][outcome_col] - control_weighted_outcome_ate)**2, 
                                 weights=data[data[treatment_col] == 0]['weight_ate'])
    se_ate = np.sqrt(var_treated_ate / len(data[data[treatment_col] == 1]) + var_control_ate / len(data[data[treatment_col] == 0]))
    ci_lower_ate = ate - 1.96 * se_ate
    ci_upper_ate = ate + 1.96 * se_ate
    
    return att, se_att, ci_lower_att, ci_upper_att, ate, se_ate, ci_lower_ate, ci_upper_ate

# 平衡表格（相同）
def balance_table_ipw(data, covariates, treatment_col, weight_col):
    balance_data = []
    for cov in covariates:
        mean_treat_before = data[data[treatment_col] == 1][cov].mean()
        mean_control_before = data[data[treatment_col] == 0][cov].mean()
        var_treat_before = data[data[treatment_col] == 1][cov].var()
        var_control_before = data[data[treatment_col] == 0][cov].var()
        pooled_sd_before = np.sqrt((var_treat_before + var_control_before) / 2)
        std_diff_before = (mean_treat_before - mean_control_before) / pooled_sd_before
        
        mean_treat_after = mean_treat_before
        control_data = data[data[treatment_col] == 0]
        mean_control_after = np.average(control_data[cov], weights=control_data[weight_col])
        var_control_after = np.average((control_data[cov] - mean_control_after)**2, 
                                       weights=control_data[weight_col])
        pooled_sd_after = np.sqrt((var_treat_before + var_control_after) / 2)
        std_diff_after = (mean_treat_after - mean_control_after) / pooled_sd_after
        
        balance_data.append({
            'Covariate': cov,
            'Mean Treated (Before)': mean_treat_before,
            'Mean Control (Before)': mean_control_before,
            'Std. Diff. Before': std_diff_before,
            'Mean Treated (After)': mean_treat_after,
            'Mean Control (After)': mean_control_after,
            'Std. Diff. After': std_diff_after
        })
    return pd.DataFrame(balance_data)

def balance_table_ipw_ate(data, covariates, treatment_col, weight_col):
    balance_data = []
    for cov in covariates:
        treated_data = data[data[treatment_col] == 1]
        control_data = data[data[treatment_col] == 0]
        mean_treat_before = treated_data[cov].mean()
        mean_control_before = control_data[cov].mean()
        var_treat_before = treated_data[cov].var()
        var_control_before = control_data[cov].var()
        pooled_sd_before = np.sqrt((var_treat_before + var_control_before) / 2)
        std_diff_before = (mean_treat_before - mean_control_before) / pooled_sd_before
        
        mean_treat_after = np.average(treated_data[cov], weights=treated_data[weight_col])
        mean_control_after = np.average(control_data[cov], weights=control_data[weight_col])
        var_treat_after = np.average((treated_data[cov] - mean_treat_after)**2, weights=treated_data[weight_col])
        var_control_after = np.average((control_data[cov] - mean_control_after)**2, weights=control_data[weight_col])
        pooled_sd_after = np.sqrt((var_treat_after + var_control_after) / 2)
        std_diff_after = (mean_treat_after - mean_control_after) / pooled_sd_after
        
        balance_data.append({
            'Covariate': cov,
            'Mean Treated (Before)': mean_treat_before,
            'Mean Control (Before)': mean_control_before,
            'Std. Diff. Before': std_diff_before,
            'Mean Treated (After)': mean_treat_after,
            'Mean Control (After)': mean_control_after,
            'Std. Diff. After': std_diff_after
        })
    return pd.DataFrame(balance_data)

# 平衡性检查（宽松版本：阈值0.2，至少90%协变量平衡）
def check_balance_ipw(data, covariates, treatment_col, weight_col):
    balance_df = balance_table_ipw(data, covariates, treatment_col, weight_col)
    balanced_ratio = (abs(balance_df['Std. Diff. After']) < 0.2).mean()
    return balanced_ratio >= 0.9, balanced_ratio  # 返回是否平衡及比率

def check_balance_ipw_ate(data, covariates, treatment_col, weight_col):
    balance_df = balance_table_ipw_ate(data, covariates, treatment_col, weight_col)
    balanced_ratio = (abs(balance_df['Std. Diff. After']) < 0.2).mean()
    return balanced_ratio >= 0.9, balanced_ratio  # 返回是否平衡及比率

# Love Plot（调整阈值到0.2，使用viridis颜色方案，并将阈值线颜色改为viridis方案一致）
def plot_balance(balance_df, model_name, effect_type='ATT', save_dir=output_dir):
    plt.figure(figsize=(12, 8))
    colors = plt.cm.viridis(np.linspace(0, 1, len(balance_df)))  # 使用viridis colormap
    for i, cov in enumerate(balance_df['Covariate']):
        before = balance_df['Std. Diff. Before'][i]
        after = balance_df['Std. Diff. After'][i]
        plt.plot([before, after], [i, i], 'o-', color=colors[i])
        plt.text(before, i, 'Before', ha='right', va='center', fontsize=8)
        plt.text(after, i, 'After', ha='left', va='center', fontsize=8)
    plt.yticks(range(len(balance_df)), balance_df['Covariate'])
    threshold_color = plt.cm.viridis(0.5)  # 使用viridis方案中的中间颜色保持一致
    plt.axvline(x=0.2, color=threshold_color, linestyle='--', label='Balance Threshold (0.2)')
    plt.axvline(x=-0.2, color=threshold_color, linestyle='--')
    plt.xlabel('Standardized Difference')
    plt.title(f'{model_name} {effect_type} Love Plot')
    plt.legend()
    plt.tight_layout()
    save_path = os.path.join(save_dir, f'{model_name}_{effect_type.lower()}_love_plot.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Love Plot 保存到: {save_path}")

# KDE图（使用viridis颜色方案）
def plot_kde_before(data, treatment_col, pscore_col, model_name, save_dir=output_dir):
    plt.figure(figsize=(10, 6))
    sns.kdeplot(data=data[data[treatment_col] == 1], x=pscore_col, label='Treated', color=plt.cm.viridis(0.2), linestyle='-')
    sns.kdeplot(data=data[data[treatment_col] == 0], x=pscore_col, label='Control', color=plt.cm.viridis(0.8), linestyle='--')
    plt.title(f'{model_name} - Propensity Score Distribution Before Weighting')
    plt.xlabel('Propensity Score')
    plt.ylabel('Density')
    plt.legend()
    plt.tight_layout()
    save_path = os.path.join(save_dir, f'{model_name}_kde_before.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"KDE Before 保存到: {save_path}")

def plot_kde_after(data, treatment_col, pscore_col, weight_col, model_name, effect_type='ATT', save_dir=output_dir):
    plt.figure(figsize=(10, 6))
    if effect_type == 'ATT':
        sns.kdeplot(data=data[data[treatment_col] == 1], x=pscore_col, label='Treated', color=plt.cm.viridis(0.2), linestyle='-')
        control_data = data[data[treatment_col] == 0]
        sns.kdeplot(data=control_data, x=pscore_col, weights=control_data[weight_col], 
                    label='Weighted Control', color=plt.cm.viridis(0.8), linestyle='--')
    else:  # ATE
        sns.kdeplot(data=data[data[treatment_col] == 1], x=pscore_col, 
                    weights=data[data[treatment_col] == 1][weight_col], 
                    label='Weighted Treated', color=plt.cm.viridis(0.2), linestyle='-')
        sns.kdeplot(data=data[data[treatment_col] == 0], x=pscore_col, 
                    weights=data[data[treatment_col] == 0][weight_col], 
                    label='Weighted Control', color=plt.cm.viridis(0.8), linestyle='--')
    plt.title(f'{model_name} - {effect_type} Propensity Score Distribution After Weighting')
    plt.xlabel('Propensity Score')
    plt.ylabel('Density')
    plt.legend()
    plt.tight_layout()
    save_path = os.path.join(save_dir, f'{model_name}_{effect_type.lower()}_kde_after.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"KDE After ({effect_type}) 保存到: {save_path}")

# 生成Word报告（添加平衡比率）
def generate_word_report(group_results, save_path, prefix="ipw_"):
    doc = Document()
    doc.add_heading('IPW分析报告', 0)
    
    for group_id, results in group_results.items():
        doc.add_heading(f'控制变量组 {group_id}', level=1)
        
        for model_name, model_res in results.items():
            doc.add_heading(f'模型 {model_name}', level=2)
            para = doc.add_paragraph()
            para.add_run(f"ATT: {model_res.get('ATT', 'N/A'):.6f}, SE: {model_res.get('SE_ATT', 'N/A'):.6f}, CI: [{model_res.get('CI Lower ATT', 'N/A'):.6f}, {model_res.get('CI Upper ATT', 'N/A'):.6f}]\n")
            para.add_run(f"ATE: {model_res.get('ATE', 'N/A'):.6f}, SE: {model_res.get('SE_ATE', 'N/A'):.6f}, CI: [{model_res.get('CI Lower ATE', 'N/A'):.6f}, {model_res.get('CI Upper ATE', 'N/A'):.6f}]\n")
            para.add_run(f"ATT Balanced: {model_res.get('ATT_Balanced', 'N/A')} (Ratio: {model_res.get('ATT_Balanced_Ratio', 'N/A'):.2f})\n")
            para.add_run(f"ATE Balanced: {model_res.get('ATE_Balanced', 'N/A')} (Ratio: {model_res.get('ATE_Balanced_Ratio', 'N/A'):.2f})")

    word_file = os.path.join(save_path, f"{prefix}analysis_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx")
    doc.save(word_file)
    print(f"Word报告已生成: {word_file}")
    return word_file

# 主程序
models = {
    'Logistic Regression': get_pscore_logistic,
    'SVM': get_pscore_svm
}

all_group_results = {}

for fold in folds:
    print(f"\n=== 处理折数: {fold} (模拟分组) ===")
    for group_id, covariates in control_groups.items():
        combo_id = f"fold_{fold}_group_{group_id}"
        print(f"\n=== 处理组合 {combo_id} ===")
        
        # 加载数据
        data = pd.read_stata(data_path)
        data = data[[treatment, outcome] + covariates].dropna()
        
        # 编码
        le_treatment = LabelEncoder()
        data['treatment_encoded'] = le_treatment.fit_transform(data[treatment])
        le_outcome = LabelEncoder()
        data['outcome_encoded'] = le_outcome.fit_transform(data[outcome])
        
        X = data[covariates].copy()
        for col in X.columns:
            if X[col].dtype == 'object' or pd.api.types.is_categorical_dtype(X[col]):
                X[col] = LabelEncoder().fit_transform(X[col].astype(str))
            X[col] = MinMaxScaler(feature_range=(0, 1)).fit_transform(X[col].values.reshape(-1, 1))
        
        for col in covariates:
            data[col] = X[col]
        
        group_results = {}
        
        for model_name, get_pscore in models.items():
            print(f"\n--- {model_name} ---")
            data_model = data.copy()
            
            try:
                data_model['pscore'] = get_pscore(X.values, data_model['treatment_encoded'].values)
                data_model['pscore'] = np.clip(data_model['pscore'], 0.01, 0.99)
                
                att, se_att, ci_lower_att, ci_upper_att, ate, se_ate, ci_lower_ate, ci_upper_ate = calculate_att_ate(
                    data_model, 'treatment_encoded', 'outcome_encoded', 'pscore')
                
                balance_df_att = balance_table_ipw(data_model, covariates, 'treatment_encoded', 'weight_att')
                all_balanced_att, balanced_ratio_att = check_balance_ipw(data_model, covariates, 'treatment_encoded', 'weight_att')
                
                balance_df_ate = balance_table_ipw_ate(data_model, covariates, 'treatment_encoded', 'weight_ate')
                all_balanced_ate, balanced_ratio_ate = check_balance_ipw_ate(data_model, covariates, 'treatment_encoded', 'weight_ate')
                
                # 保存CSV
                balance_df_att.to_csv(os.path.join(output_dir, f'{combo_id}_{model_name}_att_balance.csv'), index=False)
                balance_df_ate.to_csv(os.path.join(output_dir, f'{combo_id}_{model_name}_ate_balance.csv'), index=False)
                
                # 绘图
                plot_kde_before(data_model, 'treatment_encoded', 'pscore', f'{combo_id}_{model_name}')
                plot_kde_after(data_model, 'treatment_encoded', 'pscore', 'weight_att', f'{combo_id}_{model_name}', 'ATT')
                plot_balance(balance_df_att, f'{combo_id}_{model_name}', 'ATT')
                plot_kde_after(data_model, 'treatment_encoded', 'pscore', 'weight_ate', f'{combo_id}_{model_name}', 'ATE')
                plot_balance(balance_df_ate, f'{combo_id}_{model_name}', 'ATE')
                
                group_results[model_name] = {
                    'ATT': att, 'SE_ATT': se_att, 'CI Lower ATT': ci_lower_att, 'CI Upper ATT': ci_upper_att,
                    'ATE': ate, 'SE_ATE': se_ate, 'CI Lower ATE': ci_lower_ate, 'CI Upper ATE': ci_upper_ate,
                    'ATT_Balanced': all_balanced_att, 'ATT_Balanced_Ratio': balanced_ratio_att,
                    'ATE_Balanced': all_balanced_ate, 'ATE_Balanced_Ratio': balanced_ratio_ate
                }
                
                clear_gpu_resources()
                
            except Exception as e:
                print(f"{model_name} 处理时出错: {str(e)}")
                traceback.print_exc()
        
        all_group_results[combo_id] = group_results
        
        # 生成报告
        generate_word_report({combo_id: group_results}, output_dir)

print("所有分析完成！")

2025-10-25 09:51:11.255051: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-25 09:51:11.602602: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-25 09:51:12.997952: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


脚本开始执行...

=== 处理折数: 2 (模拟分组) ===

=== 处理组合 fold_2_group_1 ===

--- Logistic Regression ---
正在训练 cuML Logistic Regression...
KDE Before 保存到: D:\my-rapids-project\ipw_results/fold_2_group_1_Logistic Regression_kde_before.png
KDE After (ATT) 保存到: D:\my-rapids-project\ipw_results/fold_2_group_1_Logistic Regression_att_kde_after.png
Love Plot 保存到: D:\my-rapids-project\ipw_results/fold_2_group_1_Logistic Regression_att_love_plot.png
KDE After (ATE) 保存到: D:\my-rapids-project\ipw_results/fold_2_group_1_Logistic Regression_ate_kde_after.png
Love Plot 保存到: D:\my-rapids-project\ipw_results/fold_2_group_1_Logistic Regression_ate_love_plot.png

--- SVM ---
正在训练 cuML SVM...
[2025-10-25 09:51:16.586] [CUML] [warning] Random state is currently ignored by probabilistic SVC
KDE Before 保存到: D:\my-rapids-project\ipw_results/fold_2_group_1_SVM_kde_before.png
KDE After (ATT) 保存到: D:\my-rapids-project\ipw_results/fold_2_group_1_SVM_att_kde_after.png
Love Plot 保存到: D:\my-rapids-project\ipw_results/fold_2_gro